In [2]:
import torch, json
from preprocessing import build_sequences, split, SERVICES
from train import fine_tune, LORA_RANK
from score import load_finetuned, score_sequences
from evaluate import evaluate, save_results
from baseline_if import run_isolation_forest

In [ ]:
seqs = build_sequences('token')

In [ ]:
train, test = split(seqs)
fine_tune('token', train, rank=16) 

In [ ]:

model, tok = load_finetuned('token', rank=16)
scores, labels = score_sequences(test, model, tok)
evaluate(scores, labels, model_name='llm_lora', service='token')

In [ ]:
RESULTS = []

for service in SERVICES:
    print(f"\n{'='*55}\n  Service: {service}\n{'='*55}")

    # ── Data ─────────────────────────────────────────────────────────────────
    seqs = build_sequences(service)
    if not seqs:
        print("  No data — skipping.")
        continue
    train_seqs, test_seqs = split(seqs)

    # ── Isolation Forest baseline ─────────────────────────────────────────────
    print("\n  [Isolation Forest]")
    if_res = run_isolation_forest(train_seqs, test_seqs)
    if if_res:
        RESULTS.append({**if_res, "service": service, "model": "isolation_forest"})

    # ── LLM + LoRA ────────────────────────────────────────────────────────────
    print("\n  [LLM + LoRA]")
    fine_tune(service, train_seqs, rank=LORA_RANK)

    model, tokenizer = load_finetuned(service, rank=LORA_RANK)
    scores, labels   = score_sequences(test_seqs, model, tokenizer)
    llm_res          = evaluate(scores, labels,
                                model_name="llm_lora", service=service)
    if llm_res:
        RESULTS.append(llm_res)

    del model; torch.cuda.empty_cache()

# ── Save & display ────────────────────────────────────────────────────────────
save_results(RESULTS)